### Import the Data

In [ ]:
%run  Data_preparation_TCGA.ipynb

### Import the Model

In [ ]:
from Classification_model import *

### Training Process

Load the training and testing datasets

In [ ]:
train_loader = DataLoader(training_set, batch_size=1024, shuffle=True)
test_loader = DataLoader(testing_set, batch_size=5096, shuffle=False)

Initialize the DrugCell-VAE model with predefined hyperparameters and random seed.

In [ ]:
torch.manual_seed(0)

num_hiddens_genotype = 16
num_hiddens_final = 16

model = Drugcell_Vae(term_size_map, term_direct_gene_map, dG, num_genes, 
                 root, num_hiddens_genotype, num_hiddens_final, n_class = len(cancer_2_idx))

This section performs the main model training loop.  
A **term-level mask (`term_mask_map`)** is first created to constrain gradient updates of direct gene layers, ensuring that only genes directly connected to each GO term are trainable. 

In [ ]:
def create_term_mask(term_direct_gene_map, gene_dim, device):

    term_mask_map = {}

    for term, gene_set in term_direct_gene_map.items():

        mask = torch.zeros(len(gene_set), gene_dim)

        for i, gene_id in enumerate(gene_set):
            mask[i, gene_id] = 1

        mask_gpu = torch.autograd.Variable(mask)

        term_mask_map[term] = mask_gpu.to(device)

    return term_mask_map

term_mask_map = create_term_mask(model.term_direct_gene_map, num_genes, device = DEVICE)


In [ ]:
# get the paramaters from the pretrained model, and freeze them
teacher = torch.load('model_032_updated.pt')

leaves = [n for n in dG.nodes() if dG.out_degree(n) == 0]

for p in teacher.parameters():
    p.requires_grad = False

### Loading Pretrained Model Weights:  
This section loads pretrained model parameters from `model_032_updated.pt` while ensuring compatibility with the current model architecture.  
Only matching parameters (same names and tensor sizes) are loaded into the current model, and `strict=False` allows skipping unmatched layers.  
**Note:**  
Since there are no unmatched layers — even the randomly generated graph preserves the dimensional consistency of all nodes —  
the `strict` argument can be safely set to `True` without affecting model loading.

In [ ]:
model_loaded = torch.load('model_032_updated.pt', map_location='cpu')

state_dict = model_loaded.state_dict()
current_state_dict = model.state_dict()

filtered_state_dict = {
    k: v for k, v in state_dict.items()
    if k in current_state_dict and v.size() == current_state_dict[k].size()
}

model.load_state_dict(filtered_state_dict, strict=False)


### Freezing Term Modules

This function (`freeze_term_modules`) freezes the parameters of intermediate modules in the model.  
Specifically, it sets `requires_grad = False` for all parameters belonging to **linear** or **batch normalization** layers,  
except those associated with the `"final"` module, ensuring only the final layers remain trainable.

In [ ]:
def freeze_term_modules(model):
    for name, param in model.named_parameters():
        if (
            any(x in name for x in ['_linear_layer', '_batchnorm_layer'])
            and 'final' not in name
        ):
            param.requires_grad = False

### Supervised Fine-Tuning with Frozen Encoder

This section performs **supervised fine-tuning** on the pretrained model while freezing most of the encoder layers.  
Only the final classification-related modules remain trainable, allowing efficient adaptation to cancer-type prediction.

#### 1. Training Setup
- The **Adam optimizer** is configured with:
  - Learning rate: `0.003`
  - Betas: `(0.9, 0.99)`
  - Weight decay: `1e-4`
- A `term_mask_map` is created to maintain consistency during parameter updates.  
- The function `freeze_term_modules()` freezes most term-level layers except the final ones.

#### 2. Parameter Initialization
- Before training, all parameters are initialized, and direct gene layer weights are adjusted using the corresponding term masks.

#### 3. Training Loop
- Each epoch iterates through the **training loader**, performing forward and backward passes:
  - The model outputs include:
    - `logits`: classification logits  
    - `mu`, `log_var`: latent VAE components  
    - `aux_out_map`, `aux_cancer_map`: intermediate hierarchical outputs.  
        These represent the decoder results of intermediate nodes — the former corresponds to reconstruction outputs,  
        and the latter to classification outputs. If needed, they can be used for intermediate-level reconstruction (using `aux_out_map`) or classification (using `aux_cancer_map`).
  - The loss function `loss_log_vae()` returns:
    - **Classification loss**
    - **VAE regularization term**
    - **Kullback–Leibler divergence (KLD)**  
  - The overall objective minimizes the classification loss while maintaining latent structure regularization.
- Gradient updates are selectively applied based on `term_mask_map`.

#### 4. Evaluation and Monitoring
- After each epoch, the model is evaluated on the **test loader**:
  - Accuracy is computed using the predicted class labels.  
  - Loss and accuracy values are appended to tracking lists (`loss_list`, `accu_list`).  
- The `tqdm` progress bar displays the current epoch, loss, and accuracy in real time.

#### 5. Best Model Tracking
- The model achieving the highest accuracy is marked as the **best epoch**, and its weights are prepared for saving.  

**Note:**  
For debugging purposes, the model-saving function has been commented out.  
To enable model saving, remove the comment symbol (`#`) before the `torch.save` line in the code.


In [ ]:
model.to(DEVICE)
learning_rate = 0.003
torch.manual_seed(0)
loss_list = []
accu_list = []
train_epochs = 500

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.99), eps=1e-05, weight_decay = 1e-4)

term_mask_map = create_term_mask(model.term_direct_gene_map, gene_dim=num_genes, device=DEVICE)

freeze_term_modules(model)

optimizer.zero_grad()

best_epoch = 0
best_accu = 0
best_model_path = "model_classification_freeze_encoder.pt"

for name, param in model.named_parameters():
    term_name = name.split('_')[0]

    if '_direct_gene_layer.weight' in name:
        param.data = torch.mul(param.data, term_mask_map[term_name].to(DEVICE)) * 1
    else:
        param.data = param.data * 1

tepoch = tqdm.tqdm(range(train_epochs))
for epoch in tepoch:

    # Train
    model.train()
    train_predict = torch.zeros(0, 0).to(DEVICE)

    for i, (data, labels) in enumerate(train_loader):
        # Convert torch tensor to Variable

        # Forward + Backward + Optimize
        optimizer.zero_grad()  # zero the gradient buffer

        # Here term_NN_out_map is a dictionary
        logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(data.to(DEVICE))
        
        student_feats = term_NN_out_map

        if train_predict.size()[0] == 0:
            train_predict = aux_out_map["final"].data
        else:
            train_predict = torch.cat([train_predict, aux_out_map["final"].data], dim=0)

        total_loss = 0

        loss_vae, class_loss, KLD = model.loss_log_vae(
            logits=logits, y=labels.to(DEVICE), mu=mu, log_var=log_var, beta=0.001
        )

        loss_intermidiate = model.intermediate_loss_cancer(aux_cancer_map, labels.to(DEVICE))

        total_loss = torch.mean(loss_vae)

        tmp_loss = total_loss.item()
        
        total_loss.backward()

        for name, param in model.named_parameters():
            if "_direct_gene_layer.weight" not in name:
                continue
            term_name = name.split("_")[0]
            # print name, param.grad.data.size(), term_mask_map[term_name].size()
            if param.requires_grad and param.grad is not None:
                term_name = name.split("_")[0]
                param.grad.data = torch.mul(param.grad.data, term_mask_map[term_name])

        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        (inputdata, labels) = next(iter(test_loader))
        inputdata = inputdata.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(inputdata)
        preds = torch.argmax(logits, dim=1)
        accu = (preds == labels).float().mean().item()
    
        loss_list.append(tmp_loss)
        accu_list.append(accu)

    # if epoch % 10 == 0:
    if accu > best_accu:
        best_epoch = epoch
        best_accu = accu
        #torch.save(model, best_model_path)
    tepoch.set_postfix({"Epoch": epoch, "Loss": tmp_loss, "Accuracy": accu})
        
print(f"Epoch {best_epoch}: New best model saved with accuracy {best_accu:.4f}")
print("Training complete. Best model saved at:", best_model_path)


In [ ]:
"""with open('tcga_loss_list_freeze_encoder.txt', 'w') as f:
    for loss in loss_list:
        f.write(f"{loss}\n")
    

with open('tcga_accuracy_list_freeze_encoder.txt', 'w') as f:
    for loss in accu_list:
        f.write(f"{loss}\n")"""